In [ ]:
# !pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 754.1 kB/s eta 0:00:0000:0100:15
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 467.0 kB/s eta 0:00:00a 0:00:01
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813791 sha256=e0cd8cdba822476dbc2f095f08668ce073ce140cd81c0f200d173ae1e081e2f7
  Stored in directory: /Users/artemgolubnichiy/Library/Caches/pip/wheels/76/f8/dc/9195b82b8586561710077d42370ae400e8a023af43052d1fec
Successfully built pyspark


In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Titanic').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/24 16:04:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from itertools import chain
from pyspark.sql.functions import count, mean, when, lit, create_map, regexp_extract

In [3]:
df = spark.read.csv('tit_train.csv', header=True, inferSchema=True)

In [4]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [5]:
df.show(3)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
only showing top 3 rows


In [8]:
df.limit(5).toPandas()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S


In [9]:
df.select('Survived', 'Pclass', 'Age', 'Fare').show(5)

+--------+------+----+-------+
|Survived|Pclass| Age|   Fare|
+--------+------+----+-------+
|       0|     3|22.0|   7.25|
|       1|     1|38.0|71.2833|
|       1|     3|26.0|  7.925|
|       1|     1|35.0|   53.1|
|       0|     3|35.0|   8.05|
+--------+------+----+-------+
only showing top 5 rows


In [11]:
df.select('Survived', 'Pclass', 'Age', 'Fare').summary().show()

25/09/24 16:49:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------------+------------------+------------------+-----------------+
|summary|           Survived|            Pclass|               Age|             Fare|
+-------+-------------------+------------------+------------------+-----------------+
|  count|                891|               891|               714|              891|
|   mean| 0.3838383838383838| 2.308641975308642| 29.69911764705882| 32.2042079685746|
| stddev|0.48659245426485753|0.8360712409770491|14.526497332334035|49.69342859718089|
|    min|                  0|                 1|              0.42|              0.0|
|    25%|                  0|                 2|              20.0|           7.8958|
|    50%|                  0|                 3|              28.0|          14.4542|
|    75%|                  1|                 3|              38.0|             31.0|
|    max|                  1|                 3|              80.0|         512.3292|
+-------+-------------------+------------------+------

In [12]:
df.count()

891

In [14]:
len(df.columns)

12

In [15]:
df.toPandas().shape

(891, 12)

### EDA

In [16]:
df.groupBy('Survived').count().show()

+--------+-----+
|Survived|count|
+--------+-----+
|       1|  342|
|       0|  549|
+--------+-----+



In [17]:
df.groupBy('Survived').mean('Fare', 'Age').show()

+--------+------------------+------------------+
|Survived|         avg(Fare)|          avg(Age)|
+--------+------------------+------------------+
|       1| 48.39540760233917|28.343689655172415|
|       0|22.117886885245877| 30.62617924528302|
+--------+------------------+------------------+



In [18]:
df.groupBy('Survived').pivot('Sex').count().show()

+--------+------+----+
|Survived|female|male|
+--------+------+----+
|       1|   233| 109|
|       0|    81| 468|
+--------+------+----+



In [19]:
df.groupBy('Survived').pivot('Pclass').count().show()

+--------+---+---+---+
|Survived|  1|  2|  3|
+--------+---+---+---+
|       1|136| 87|119|
|       0| 80| 97|372|
+--------+---+---+---+



In [20]:
df.groupBy('Survived').pivot('SibSp').count().show()

+--------+---+---+---+---+---+----+----+
|Survived|  0|  1|  2|  3|  4|   5|   8|
+--------+---+---+---+---+---+----+----+
|       1|210|112| 13|  4|  3|NULL|NULL|
|       0|398| 97| 15| 12| 15|   5|   7|
+--------+---+---+---+---+---+----+----+



In [24]:
for col in df.columns:
    print(col.ljust(20), df.filter(df[col].isNull()).count())

PassengerId          0
Survived             0
Pclass               0
Name                 0
Sex                  0
Age                  177
SibSp                0
Parch                0
Ticket               0
Fare                 0
Cabin                687
Embarked             0


In [22]:
df.select('Fare', 'Embarked').summary('mean', '50%', 'max').show()

+-------+----------------+--------+
|summary|            Fare|Embarked|
+-------+----------------+--------+
|   mean|32.2042079685746|    NULL|
|    50%|         14.4542|    NULL|
|    max|        512.3292|       S|
+-------+----------------+--------+



In [23]:
df = df.fillna({'Embarked': 'S'})

### Извлечение информации о титулах

In [25]:
df = df.withColumn('Title', regexp_extract(df['Name'], '([A-Za-z]+)\.', 1))
df.groupBy('Title').agg(count('Age'), mean('Age')).sort('count(Age)').show()

+--------+----------+------------------+
|   Title|count(Age)|          avg(Age)|
+--------+----------+------------------+
|     Don|         1|              40.0|
|Countess|         1|              33.0|
|    Lady|         1|              48.0|
|     Mme|         1|              24.0|
|    Capt|         1|              70.0|
|     Sir|         1|              49.0|
|Jonkheer|         1|              38.0|
|      Ms|         1|              28.0|
|     Col|         2|              58.0|
|    Mlle|         2|              24.0|
|   Major|         2|              48.5|
|     Rev|         6|43.166666666666664|
|      Dr|         6|              42.0|
|  Master|        36| 4.574166666666667|
|     Mrs|       108|35.898148148148145|
|    Miss|       146|21.773972602739725|
|      Mr|       398|32.368090452261306|
+--------+----------+------------------+



In [27]:
title_dic = {'Mr':'Mr', 'Miss':'Miss', 'Mrs':'Mrs', 'Master':'Master', \
             'Mlle': 'Miss', 'Major': 'Mr', 'Col': 'Mr', 'Sir': 'Mr',\
             'Don': 'Mr', 'Mme': 'Miss', 'Jonkheer': 'Mr', 'Lady': 'Mrs',\
             'Capt': 'Mr', 'Countess': 'Mrs', 'Ms': 'Miss', 'Dona': 'Mrs', \
             'Dr':'Mr', 'Rev':'Mr'}

In [28]:
mapping = create_map([lit(x) for x in chain(*title_dic.items())])
df = df.withColumn('Title', mapping[df['Title']])
df.groupBy('Title').mean('Age').show()

+------+------------------+
| Title|          avg(Age)|
+------+------------------+
|  Miss|             21.86|
|Master| 4.574166666666667|
|    Mr| 33.02272727272727|
|   Mrs|35.981818181818184|
+------+------------------+



In [29]:
def age_imputer(df, title, age):
    return df.withColumn('Age', when((df['Age'].isNull()) & (df['Title'] == title), age).otherwise(df['Age']))


In [30]:
df = age_imputer(df, 'Miss', 21.86)
df = age_imputer(df, 'Master', 4.57)
df = age_imputer(df, 'Mr', 33.02)
df = age_imputer(df, 'Mrs', 35.98)

In [31]:
df.show()

+-----------+--------+------+--------------------+------+-----+-----+-----+----------------+-------+-----+--------+------+
|PassengerId|Survived|Pclass|                Name|   Sex|  Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked| Title|
+-----------+--------+------+--------------------+------+-----+-----+-----+----------------+-------+-----+--------+------+
|          1|       0|     3|Braund, Mr. Owen ...|  male| 22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|    Mr|
|          2|       1|     1|Cumings, Mrs. Joh...|female| 38.0|    1|    0|        PC 17599|71.2833|  C85|       C|   Mrs|
|          3|       1|     3|Heikkinen, Miss. ...|female| 26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|  Miss|
|          4|       1|     1|Futrelle, Mrs. Ja...|female| 35.0|    1|    0|          113803|   53.1| C123|       S|   Mrs|
|          5|       0|     3|Allen, Mr. Willia...|  male| 35.0|    0|    0|          373450|   8.05| NULL|       S|    Mr|
|          6|   

In [33]:
df = df.drop('PassengerId', 'Cabin', 'Name', 'Ticket', 'Title')

In [34]:
for col in df.columns:
    print(col.ljust(20), df.filter(df[col].isNull()).count())

Survived             0
Pclass               0
Sex                  0
Age                  0
SibSp                0
Parch                0
Fare                 0
Embarked             0


### Построение модели (Spark ML)

- StringIndexer: преобразование строковых категорий в численные
- Векторный Ассемблер: Spark API
- Логистическая регрессия на основе регуляризации Ridge Lasso
- Древовидные ансамблевые методы: random forest
- GBT
- Конвейер

In [35]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

### String Indexer

In [36]:
stringIndex = StringIndexer(inputCols=['Sex', 'Embarked'], outputCols=['SexNum', 'EmbarkedNum'])
stringIndex_model = stringIndex.fit(df)
df_ = stringIndex_model.transform(df).drop('Sex', 'Embarked')
df_.show(3)

+--------+------+----+-----+-----+-------+------+-----------+
|Survived|Pclass| Age|SibSp|Parch|   Fare|SexNum|EmbarkedNum|
+--------+------+----+-----+-----+-------+------+-----------+
|       0|     3|22.0|    1|    0|   7.25|   0.0|        0.0|
|       1|     1|38.0|    1|    0|71.2833|   1.0|        1.0|
|       1|     3|26.0|    0|    0|  7.925|   1.0|        0.0|
+--------+------+----+-----+-----+-------+------+-----------+
only showing top 3 rows


### Векторный ассемблер

In [37]:
vec_asmbl = VectorAssembler(inputCols=df_.columns[1:], outputCol='features')
df_ = vec_asmbl.transform(df_).select('features', 'Survived')
df_.show(3, truncate=False)

+----------------------------------+--------+
|features                          |Survived|
+----------------------------------+--------+
|[3.0,22.0,1.0,0.0,7.25,0.0,0.0]   |0       |
|[1.0,38.0,1.0,0.0,71.2833,1.0,1.0]|1       |
|[3.0,26.0,0.0,0.0,7.925,1.0,0.0]  |1       |
+----------------------------------+--------+
only showing top 3 rows


### Разбивка данных

In [38]:
train_df, valid_df = df_.randomSplit([0.7, 0.3])
train_df.show(3, truncate=False)

+---------------------+--------+
|features             |Survived|
+---------------------+--------+
|(7,[0,1],[1.0,33.02])|0       |
|(7,[0,1],[1.0,33.02])|0       |
|(7,[0,1],[1.0,38.0]) |0       |
+---------------------+--------+
only showing top 3 rows


### Вычисление и метрики

In [ ]:
evalutor = MulticlassClassificationEvaluator(labelCol='Survived', metricName='accuracy')

### Логистическая регрессия

In [43]:
ridge = LogisticRegression(labelCol='Survived', maxIter=100, elasticNetParam=0, regParam=0.3)

model = ridge.fit(train_df)
pred = model.transform(valid_df)
evalutor.evaluate(pred)

0.8066914498141264

### Случайный лес

In [47]:
rf = RandomForestClassifier(labelCol='Survived', numTrees=100, maxDepth=4)

model = rf.fit(train_df)
pred = model.transform(valid_df)
evalutor.evaluate(pred)

0.8215613382899628

### GBT

In [49]:
gb = GBTClassifier(labelCol='Survived', maxIter=200, maxDepth=4)

model = gb.fit(train_df)
pred = model.transform(valid_df)
evalutor.evaluate(pred)

25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1000.4 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1000.9 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1001.5 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1002.7 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1005.0 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1005.5 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1006.1 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1007.3 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1009.6 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1010.1 KiB
25/09/24 18:21:31 WARN DAGScheduler: Broadcasting large task binary with size 1010.7 KiB
25/09/24 18:21:31 WAR

0.8178438661710037

### Grid-search CV

In [50]:
pipeline_rf = Pipeline(stages=[stringIndex, vec_asmbl, rf])
paramGrid = ParamGridBuilder().addGrid(rf.maxDepth, [3, 4, 5]).addGrid(rf.minInfoGain, [0, 0.01, 0.1]).addGrid(rf.numTrees, [100, 200, 500]).build()
selected_model = CrossValidator(estimator=pipeline_rf, estimatorParamMaps=paramGrid, evaluator=evalutor, numFolds=5)
model_final = selected_model.fit(df)
pred_train = model_final.transform(df)
evalutor.evaluate(pred_train)

25/09/24 18:26:59 WARN DAGScheduler: Broadcasting large task binary with size 1583.8 KiB
25/09/24 18:27:03 WARN DAGScheduler: Broadcasting large task binary with size 1563.1 KiB
25/09/24 18:27:06 WARN DAGScheduler: Broadcasting large task binary with size 1254.4 KiB
25/09/24 18:27:09 WARN DAGScheduler: Broadcasting large task binary with size 1143.1 KiB
25/09/24 18:27:10 WARN DAGScheduler: Broadcasting large task binary with size 1919.6 KiB
25/09/24 18:27:14 WARN DAGScheduler: Broadcasting large task binary with size 1816.3 KiB
25/09/24 18:27:18 WARN DAGScheduler: Broadcasting large task binary with size 1254.4 KiB
25/09/24 18:27:20 WARN DAGScheduler: Broadcasting large task binary with size 1024.2 KiB
25/09/24 18:27:20 WARN DAGScheduler: Broadcasting large task binary with size 1143.1 KiB
25/09/24 18:27:21 WARN DAGScheduler: Broadcasting large task binary with size 2017.7 KiB
25/09/24 18:27:22 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
25/09/24 18:27:25 WARN D

0.8507295173961841